# Matched Molecular Series With OEMMPA

A matched molecular pair compares two molecules. A *matched molecular series*
compares many: one shared scaffold, several R-groups at the same attachment
points, and a measured value for each. That shape is what lets you ask the
question a pair cannot answer -- **which substituent should I try next?**

This notebook walks through the series workflow:

1. Build a persistent DuckDB store, which is where series live.
2. Read the series back and pick a scaffold worth studying.
3. Rank candidate substituents against an observed trend (the Matsy method).
4. Find other series whose R-groups behave the same way.
5. Build the all-pairs similarity network over every series at once.
6. Pool series by attachment environment so near-identical scaffolds answer together.

It expects an environment with `oemmpa`, `pandas` and `openeye-toolkits`.
Every result object here also answers `to_dataframe(molecules=True)` for
inline depictions if you have
[`cnotebook`](https://github.com/scott-arne/cnotebook) installed.

If you have not met OEMMPA's pair analysis yet, start with
[`getting_started.ipynb`](getting_started.ipynb) -- this notebook assumes you
know what a constant region and a transform are.

## Key Concepts

<dl>

  <dt><b>Matched molecular series</b></dt>
  <dd>One constant region plus the set of R-groups observed on it. A series of
  two members is just a matched pair; the interesting ones have many.</dd>

  <dt><b>R-group</b></dt>
  <dd>The variable fragment attached at a labelled point, written as SMILES
  with the attachment shown: <tt>[*:1]Cl</tt>.</dd>

  <dt><b>Cut count</b></dt>
  <dd>How many attachment points the constant has. Series are only comparable
  to other series of the same cut count.</dd>

  <dt><b><tt>rank_next_substituents</tt></b></dt>
  <dd>The Matsy method. Given R-groups ordered by a measured value, it finds
  stored series whose members are ordered the same way and reports what those
  series contain that yours does not.</dd>

  <dt><b><tt>find_similar_series</tt></b></dt>
  <dd>Series whose shared R-groups rank in the same order as the query's,
  scored by Spearman correlation and centred RMSD.</dd>

  <dt><b><tt>series_network</tt></b></dt>
  <dd>The same comparison run over every pair of series at once, returned as
  undirected edges -- a map of which scaffolds behave alike.</dd>

  <dt><b>Environment radius</b></dt>
  <dd>Pools series whose attachment environments match out to a given radius,
  so two scaffolds differing far from the attachment answer as one.</dd>

</dl>

## Setup

In [ ]:
import tempfile
from pathlib import Path

import oemmpa
import pandas as pd


## The Dataset

We use the cannabinoid receptor subset of the bundled Bender data: 99
compounds with pIC50 values. One assay, one target -- series compare values
to each other, so mixing assays would compare numbers that do not belong on
the same axis.

In [ ]:
frame = pd.read_csv("data/Bender.csv")
cannabinoid = frame[frame["Target"] == "Cannabinoid"]
print(f"{len(cannabinoid)} compounds")
cannabinoid.head()

## Build A Store

Pairs can be analysed entirely in memory, but **series are read from a store**:
enumerating them is part of saving, not of analysing. So the workflow is
analyse, then save, then query.

The fragmentation method always enumerates series, so no extra flag is needed
here. We write to a temporary directory to keep re-running this notebook
clean; in real use you would pick a path and keep it.

In [ ]:
analyzer = oemmpa.Analyzer()
for row in cannabinoid.itertuples():
    analyzer.add_molecule(row.Molecule, id=row.Title)
    analyzer.add_property(row.Title, "pIC50", float(row.pIC50))

analyzer.analyze()

store_path = Path(tempfile.mkdtemp()) / "cannabinoid.duckdb"
store = oemmpa.DuckDBStore(str(store_path))
store.save_analyzer(analyzer)

store.summary()

`series` counts the stored series and `series_members` the member rows -- one
per R-group realised by a compound. `series_variant_members` counts members
contributed by up-enumerated stereoisomers, which every read ignores unless
you ask for them with `include_up_enumerated=True`.

## What Series Do We Have?

`store.series()` returns one row per member. Grouping by series gives the
scaffolds, and the ones worth looking at have a substantial constant and
several members.

In [ ]:
members = store.series().to_dataframe()
print(f"{members.series_id.nunique()} series, {len(members)} member rows")

scaffolds = (
    members.groupby(["series_id", "constant", "cut_count", "num_members", "constant_num_heavies"])
    .size()
    .reset_index(name="rows")
)
# Most series are small or sit on a trivial constant -- a lone chlorine is a
# "series" too. Keep the ones with a real scaffold and enough members to show
# a trend.
scaffolds = scaffolds[
    (scaffolds.constant_num_heavies >= 12) & (scaffolds.num_members >= 5)
].sort_values(["num_members", "constant_num_heavies"], ascending=False)
scaffolds.head(5)

The largest is a two-cut scaffold carrying a chlorophenyl and a
dichlorophenyl, with dozens of different central rings hung between them.
That is a genuine series: one place to vary, many things tried there.

In [ ]:
series_id = int(scaffolds.iloc[0].series_id)
members[members.series_id == series_id][["rgroup", "compound_id"]].head(10)

## Rank The Next Substituent

The Matsy question: *given what I have measured so far, what should I make
next?* You hand it R-groups with their values, in any order; it sorts them,
finds stored series whose members are ordered the same way, and reports what
those series contain that yours does not.

`wins` counts the matched series where the candidate beat your best member,
`support` how many matched series contained it at all, and `fraction` their
ratio -- so a candidate with `fraction` 1.0 improved on your best every time
it was seen.

The `support_ladder` is the bar a candidate's support must clear, tried from
left to right. The default `(20, 10, 5)` suits a corporate corpus; a
99-compound demonstration set has nothing like that depth, so we lower it.
Treat the numbers below as an illustration of the mechanics, not as evidence
about cannabinoid chemistry.

In [ ]:
ranking = oemmpa.rank_next_substituents(
    store,
    "pIC50",
    "higher",
    rgroups={"[*:1]C": 6.0, "[*:1]CC": 6.5, "[*:1]CCC": 7.0},
    support_ladder=(3, 2, 1),
)

print(f"support level {ranking.support_level}, matched {ranking.matched_series} series")
ranking.to_dataframe()

Pass `evidence=True` to get the matched series themselves rather than only
the tally, which is what you want before believing any of it:

In [ ]:
evidenced = oemmpa.rank_next_substituents(
    store,
    "pIC50",
    "higher",
    rgroups={"[*:1]C": 6.0, "[*:1]CC": 6.5, "[*:1]CCC": 7.0},
    support_ladder=(3, 2, 1),
    evidence=True,
)
evidenced.evidence.to_dataframe()

## Find Series That Behave The Same Way

A different question: *has this structure-activity relationship been seen
elsewhere?* `find_similar_series` takes a stored series and returns the others
whose shared R-groups rank in the same order.

`overlap` is how many R-groups the two series have in common -- the comparison
rests entirely on those, so a high correlation over an overlap of three is a
weaker claim than the same number over ten. `spearman` is the rank
correlation; `centred_rmsd` compares the values after removing each series'
mean, so it measures disagreement in size rather than in level.

In [ ]:
similar = oemmpa.find_similar_series(store, "pIC50", series_id=series_id, top=5)
similar.to_dataframe()

## The Whole Network At Once

`series_network` runs that comparison over every pair of series and returns
undirected edges. It is the map view: instead of asking about one scaffold,
you get every relationship the store supports.

`top` keeps each series' strongest few neighbours before the edges are
unioned, which keeps the result readable on a dense store. The work is
parallel -- pass `threads` to use more of them.

In [ ]:
network = oemmpa.series_network(store, "pIC50", top=5, threads=4)
edges = network.to_dataframe()
print(f"{len(edges)} edges")
edges.sort_values(["spearman", "overlap"], ascending=False).head(10)

## Pool By Attachment Environment

Two scaffolds that differ only far from the attachment point often behave
identically, yet they are separate series and each carries only its own few
members. `environment_radius` pools series whose attachment environments match
out to that radius, so they answer as one and the evidence adds up.

The radius is a dial between two extremes. A large radius demands that most of
the surroundings match, so little pools and the result approaches the unpooled
network. A small radius demands almost nothing, so series merge wholesale into
a handful of large profiles -- and at radius 0, which pools on the attachment
atom alone, so few distinct profiles are left that no pair of them shares
enough R-groups to produce an edge at all.

Pooling is therefore not a free win: it buys depth per unit by having fewer
units. Here is the whole dial on this store:

In [ ]:
for radius in (None, 0, 1, 2, 3):
    pooled = oemmpa.series_network(
        store, "pIC50", top=5, threads=4, environment_radius=radius
    ).to_dataframe()
    nodes = len(set(pooled.source_id) | set(pooled.target_id)) if len(pooled) else 0
    label = "no pooling" if radius is None else f"radius {radius}"
    print(f"{label:>11}: {len(pooled):4d} edges over {nodes:4d} nodes")

`environment_radius` is accepted by `rank_next_substituents` and
`find_similar_series` too, and means the same thing in all three: the unit of
comparison becomes the pooled profile rather than the stored series.

## The Same Thing From The Command Line

Every query above has a CLI equivalent that works against a store built by
`oemmpa build`:

```bash
oemmpa build --smiles compounds.smi --properties values.csv --output store.duckdb
oemmpa series store.duckdb
oemmpa series-rank store.duckdb --property pIC50 --direction higher --series-id 493
oemmpa series-similar store.duckdb --property pIC50 --series-id 493 --top 5
oemmpa series-network store.duckdb --property pIC50 --top 5 --environment-radius 1
```

See [the CLI reference](../docs/cli.md) for the full set of flags.

## Where To Next

- [`getting_started.ipynb`](getting_started.ipynb) -- matched pairs,
  objectives, and generating products from observed transforms.
- [Python API reference](../docs/python-api.md) -- every option on the three
  series queries, including `selection`, `min_overlap`, `measure` and
  `include_up_enumerated`.
- [Quickstart](../docs/quickstart.md) -- the same ground in a shorter form,
  with the store schema.
- [CLI reference](../docs/cli.md) -- `series`, `series-rank`,
  `series-similar` and `series-network`.